In [1]:
import os

In [13]:
pasta_raw_objeto = "/home/likeableng/BeACoN/arquivos_de_reducao/25jul19/raw/h167128/"
pasta_raw_calib = "/home/likeableng/BeACoN/arquivos_de_reducao/25jul19/raw/calib/"

In [8]:
mapeamento = {
    'F1': 'b',
    'F2': 'v',
    'F3': 'r',
    'F4': 'i'
}

In [9]:
def ajustar_nome_filtro(caminho_da_pasta):
# Percorre todos os arquivos dentro da pasta especificada
    for nome_arquivo in os.listdir(caminho_da_pasta):
        if nome_arquivo.endswith('.fits'):
            for chave, letra in mapeamento.items():
                marcador_antigo = f'_{chave}_'
                marcador_novo = f'_{letra}_'
            
                if marcador_antigo in nome_arquivo:
                    # É preciso juntar o caminho da pasta com o nome do arquivo
                    caminho_antigo = os.path.join(caminho_da_pasta, nome_arquivo)
                    novo_nome = nome_arquivo.replace(marcador_antigo, marcador_novo)
                    caminho_novo = os.path.join(caminho_da_pasta, novo_nome)
                
                    os.rename(caminho_antigo, caminho_novo)
                    print(f'{nome_arquivo}  ->  {novo_nome}')
                    break


In [14]:
ajustar_nome_filtro(pasta_raw_calib)


flat_g5_F1_001_055.fits  ->  flat_g5_b_001_055.fits
flat_F4_004_122.fits  ->  flat_i_004_122.fits
flat_F2_002_058.fits  ->  flat_v_002_058.fits
flat_g5_F1_001_024.fits  ->  flat_g5_b_001_024.fits
flat_F1_001_195.fits  ->  flat_b_001_195.fits
flat_F4_004_113.fits  ->  flat_i_004_113.fits
flat_F2_002_035.fits  ->  flat_v_002_035.fits
flat_F4_004_182.fits  ->  flat_i_004_182.fits
flat_g5_F2_002_146.fits  ->  flat_g5_v_002_146.fits
flat_g5_F1_001_034.fits  ->  flat_g5_b_001_034.fits
flat_g5_F3_003_169.fits  ->  flat_g5_r_003_169.fits
flat_F2_002_129.fits  ->  flat_v_002_129.fits
flat_g5_F4_004_122.fits  ->  flat_g5_i_004_122.fits
flat_F4_004_031.fits  ->  flat_i_004_031.fits
flat_F4_004_104.fits  ->  flat_i_004_104.fits
flat_F1_001_187.fits  ->  flat_b_001_187.fits
flat_F3_003_147.fits  ->  flat_r_003_147.fits
flat_F1_001_072.fits  ->  flat_b_001_072.fits
flat_g5_F2_002_031.fits  ->  flat_g5_v_002_031.fits
flat_g5_F2_002_063.fits  ->  flat_g5_v_002_063.fits
flat_F1_001_054.fits  ->  flat_b

In [11]:
ajustar_nome_filtro(pasta_raw_objeto)

h167128_F3_007_proc.fits  ->  h167128_r_007_proc.fits
h167128_F4_003_proc.fits  ->  h167128_i_003_proc.fits
h167128_F4_004.fits  ->  h167128_i_004.fits
h167128_F2_015.fits  ->  h167128_v_015.fits
h167128_F1_005_proc.fits  ->  h167128_b_005_proc.fits
h167128_F1_001_proc.fits  ->  h167128_b_001_proc.fits
h167128_F2_001_proc.fits  ->  h167128_v_001_proc.fits
h167128_F1_010_proc.fits  ->  h167128_b_010_proc.fits
h167128_F1_013_proc.fits  ->  h167128_b_013_proc.fits
h167128_F2_011_proc.fits  ->  h167128_v_011_proc.fits
h167128_F1_012.fits  ->  h167128_b_012.fits
h167128_F2_013_proc.fits  ->  h167128_v_013_proc.fits
h167128_F3_001.fits  ->  h167128_r_001.fits
h167128_F1_016_proc.fits  ->  h167128_b_016_proc.fits
h167128_F2_008_proc.fits  ->  h167128_v_008_proc.fits
h167128_F2_014.fits  ->  h167128_v_014.fits
h167128_F3_004_proc.fits  ->  h167128_r_004_proc.fits
h167128_F3_009_proc.fits  ->  h167128_r_009_proc.fits
h167128_F4_001_proc.fits  ->  h167128_i_001_proc.fits
h167128_F3_011.fits  -> 

In [19]:
import os
import warnings
from collections import defaultdict
from astropy import units as u
from astropy.io import fits
from astropy.utils.exceptions import AstropyWarning

from astropop.image import imcombine, processing, imarith
from astropop.framedata import read_framedata
from astropop.file_collection import FitsFileGroup

warnings.simplefilter('ignore', category=AstropyWarning)
pasta_calib = '/home/likeableng/BeACoN/arquivos_de_reducao/25jul19/raw/calib/'
pasta_calib_reduced = '/home/likeableng/BeACoN/arquivos_de_reducao/25jul19/reduced/calib/'

letra_filtro = ["", "b", "v", "r", "i"]

def criar_master_flat(pasta_calib, pasta_calib_reduced):
    main_fg = FitsFileGroup(location=pasta_calib, fits_ext=['.fits'], ext=0)
    flat_files = main_fg.filtered({'OBJECT': 'flat'}).files

    # 1. Agrupamento manual duplo (por Filtro e Preamp limpo)
    arquivos_por_grupo = defaultdict(list)
    for arq in flat_files:
        hdr = fits.getheader(arq)
        if hdr.get('NAXIS') == 2:
            # Extrai e limpa o PREAMP (ex: '5.1x' vira '5')
            preamp_raw = str(hdr.get('PREAMP', 'UNKNOWN'))
            preamp_val = preamp_raw.lower().replace('x', '').split('.')[0]
            
            # Extrai o filtro
            filtro_val = str(hdr.get('FILTER', 'UNKNOWN'))
            
            # Agrupa os dois juntos
            arquivos_por_grupo[(filtro_val, preamp_val)].append(arq)

    # 2. Processa cada grupo separadamente
    for (filtro_val, preamp_val), arquivos_flat_2d in arquivos_por_grupo.items():
        print(f"\n--- Processando Master Flat | Filtro: {filtro_val} | PREAMP: {preamp_val} ({len(arquivos_flat_2d)} arquivos) ---")
        
        # Define sufixo e carrega o Bias correto
        sufixo_preamp = f"_g{preamp_val}" if preamp_val != '1' else ""
        caminho_master_bias = os.path.join(pasta_calib_reduced, f'master_bias{sufixo_preamp}.fits')
        
        if not os.path.exists(caminho_master_bias):
            print(f"ERRO: {caminho_master_bias} não encontrado! Abortando este flat.")
            continue
            
        master_bias = read_framedata(caminho_master_bias)

        flat_frames = [read_framedata(f, unit='adu', use_memmap_backend=True) for f in arquivos_flat_2d]
        gain = float(flat_frames[0].header['GAIN']) * u.electron / u.adu

        for f_idx, frame in enumerate(flat_frames):
            print(f" -> Processando flat [{filtro_val}] {f_idx+1}/{len(flat_frames)}", end='\r')
            processing.gain_correct(frame, gain, inplace=True)
            processing.subtract_bias(frame, master_bias, inplace=True)

        print(f"\nCombinando e normalizando Master Flat ({filtro_val}, PREAMP {preamp_val})...")
        master_flat = imcombine(flat_frames, method='median')
        mean_value = master_flat.mean()
        master_flat = imarith(master_flat, mean_value, '/')

        master_flat.meta['astropop master_type'] = 'flat'
        master_flat.meta['FILTER'] = filtro_val
        master_flat.meta['PREAMP'] = preamp_val

        valor_filtro = filtro_val[1:]
        numero_filtro = int(valor_filtro)

        # Salva com o nome correto
        nome_out = f'master_flat{sufixo_preamp}_{letra_filtro[numero_filtro]}.fits'
        
        mflat_path = os.path.join(pasta_calib_reduced, nome_out)
        master_flat.write(mflat_path, overwrite=True, no_fits_standard_units=True)

        print(f"Salvo: {mflat_path}")
        del flat_frames

criar_master_flat(pasta_calib, pasta_calib_reduced)


--- Processando Master Flat | Filtro: F1 | PREAMP: 1 (200 arquivos) ---
 -> Processando flat [F1] 200/200
Combinando e normalizando Master Flat (F1, PREAMP 1)...


2026-10-13 09:55:57,220 astropop - INFO - Combining 200 images with median method.  [imcombine]


Salvo: /home/likeableng/BeACoN/arquivos_de_reducao/25jul19/reduced/calib/master_flat_b.fits

--- Processando Master Flat | Filtro: F1 | PREAMP: 5 (200 arquivos) ---
 -> Processando flat [F1] 200/200
Combinando e normalizando Master Flat (F1, PREAMP 5)...


2026-10-13 09:56:10,462 astropop - INFO - Combining 200 images with median method.  [imcombine]


Salvo: /home/likeableng/BeACoN/arquivos_de_reducao/25jul19/reduced/calib/master_flat_g5_b.fits

--- Processando Master Flat | Filtro: F4 | PREAMP: 5 (200 arquivos) ---
 -> Processando flat [F4] 200/200
Combinando e normalizando Master Flat (F4, PREAMP 5)...


...

Salvo: /home/likeableng/BeACoN/arquivos_de_reducao/25jul19/reduced/calib/master_flat_i.fits

--- Processando Master Flat | Filtro: F3 | PREAMP: 1 (200 arquivos) ---
 -> Processando flat [F3] 200/200
Combinando e normalizando Master Flat (F3, PREAMP 1)...


2026-10-13 09:57:19,264 astropop - INFO - Combining 200 images with median method.  [imcombine]


Salvo: /home/likeableng/BeACoN/arquivos_de_reducao/25jul19/reduced/calib/master_flat_r.fits

--- Processando Master Flat | Filtro: F2 | PREAMP: 1 (200 arquivos) ---
 -> Processando flat [F2] 200/200
Combinando e normalizando Master Flat (F2, PREAMP 1)...


2026-10-13 09:57:32,987 astropop - INFO - Combining 200 images with median method.  [imcombine]


Salvo: /home/likeableng/BeACoN/arquivos_de_reducao/25jul19/reduced/calib/master_flat_v.fits


In [ ]:
letra_filtro = ["", "b", "v", "r", "i"]

print(letra_filtro[1])

b


In [18]:
letra_filtro = ["", "b", "v", "r", "i"]

filtro_val =  "F1"
valor_filtro = filtro_val[1:]
inteiro = int(valor_filtro)
print(valor_filtro)

print(letra_filtro[inteiro])

1
b
